# Gravity-Magnetic Joint Inversion Training Notebook

Reproducing Fang et al., "Improved 3-D Joint Inversion of Gravity and Magnetic Data
Based on Deep Learning With a Multitask Learning Strategy",
IEEE TGRS, Vol. 63, 2025

**Run cells sequentially.** Each cell is self-contained and can be re-run independently.

In [ ]:
# Cell 1: Import dependencies, set random seed, detect GPU
import os
import sys
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import set_seed, get_device, gpu_memory_summary, save_json, setup_logger
from src.model.joint_inversion_net import JointInversionNet
from src.model.loss_functions import get_criterion
from src.data.dataset import create_dataloaders
from src.evaluate import compute_all_metrics

# ---- Reproducibility ----
SEED = 42
set_seed(SEED)
print(f"Random seed set to {SEED}")

# ---- Device ----
device = get_device()
print(f"Using device: {device}")
if device.type == 'cuda':
    print(gpu_memory_summary())

In [ ]:
# Cell 2: Load dataset
DATA_DIR = 'data'          # directory containing .npz files + index JSONs
BATCH_SIZE = 16           # adjust based on GPU memory (RTX 5000 Ada 32GB)
NUM_WORKERS = 4

print(f"Loading data from: {DATA_DIR}")
train_loader, val_loader, test_loader = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f"Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_loader.dataset)} samples, {len(val_loader)} batches")
print(f"Test:  {len(test_loader.dataset)} samples, {len(test_loader)} batches")

# Inspect one sample
sample = next(iter(train_loader))
print(f"\nSample keys: {list(sample.keys())}")
print(f"Input shape:      {sample['input'].shape}")
print(f"Density shape:    {sample['density'].shape}")
print(f"Suscept shape:    {sample['susceptibility'].shape}")
print(f"StructSim shape:  {sample['structural_sim'].shape}")

In [ ]:
# Cell 3: Define model (JointInversionNet)
model = JointInversionNet(
    in_channels=2,
    backbone_channels=64,
    aspp_out_channels=40,
    leaky_slope=0.01,
).to(device)

param_info = model.get_num_params()
print("Model parameter counts:")
for k, v in param_info.items():
    print(f"  {k:>15s}: {v:>12,}")

# Quick forward pass test -- input is (B, 2, 81, 81) observation surface
x_dummy = torch.randn(2, 2, 81, 81, device=device)
with torch.no_grad():
    out_test = model(x_dummy)
print("\nForward pass test:")
for k, v in out_test.items():
    print(f"  {k}: {v.shape}  range=[{v.min().item():.4f}, {v.max().item():.4f}]")

In [ ]:
# Cell 4: Define loss function
criterion = get_criterion(
    task_weights={
        'task1': 1.0,  # Independent gravity (MSE)
        'task2': 1.0,  # Independent magnetic (MSE)
        'task3': 1.0,  # Structural similarity (BCE)
        'task4': 1.0,  # Joint gravity (MSE)
        'task5': 1.0,  # Joint magnetic (MSE)
    },
    leaky_relu_lambda=1e-5,  # L2 weight penalty
)
print("Loss function configured:")
print("  Tasks 1,2,4,5: MSELoss (regression)")
print("  Task 3:         BCEWithLogitsLoss (classification)")
print(f"  L2 regularization lambda: 1e-5")

In [ ]:
# Cell 5: Define optimizer (Adam + adaptive LR)
LR = 1e-3
BETAS = (0.9, 0.999)
WEIGHT_DECAY = 1e-5
EPOCHS = 90

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing LR scheduler (smooth decay over all epochs)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

# AMP scaler for mixed precision training
scaler = GradScaler(enabled=(device.type == 'cuda'))

GRAD_CLIP = 1.0  # gradient clipping max norm

print(f"Optimizer: Adam(lr={LR}, betas={BETAS}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR(T_max={EPOCHS})")
print(f"AMP enabled: {scaler.is_enabled()}")
print(f"Gradient clipping: max_norm={GRAD_CLIP}")

In [ ]:
# Cell 6: Training loop (with validation each epoch)
OUTPUT_DIR = 'results'
os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)

best_val_loss = float('inf')
history = {'train': [], 'val': []}
early_stop_counter = 0
EARLY_STOP_PATIENCE = 15
SAVE_EVERY = 10  # save checkpoint every N epochs
history_path = os.path.join(OUTPUT_DIR, 'training_history.json')

print(f"Starting training for {EPOCHS} epochs...")
print("=" * 80)
total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    # ---- Train ----
    model.train()
    train_task_losses = {f'task{i}': 0.0 for i in range(1, 6)}
    train_total_loss = 0.0
    n_train_batches = 0

    for batch in train_loader:
        inputs = batch['input'].to(device)
        targets = {
            'density': batch['density'].unsqueeze(1).to(device),
            'susceptibility': batch['susceptibility'].unsqueeze(1).to(device),
            'structural_sim': batch['structural_sim'].unsqueeze(1).to(device),
        }

        optimizer.zero_grad()
        
        with autocast(enabled=scaler.is_enabled()):
            preds = model(inputs)
            per_task, total_loss = criterion(preds, targets, model=model)
        
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        for i in range(1, 6):
            train_task_losses[f'task{i}'] += (
                per_task[f'task{i}'].item() if isinstance(per_task[f'task{i}'], torch.Tensor)
                else per_task[f'task{i}']
            )
        train_total_loss += total_loss.item()
        n_train_batches += 1

    for i in range(1, 6):
        train_task_losses[f'task{i}'] /= max(n_train_batches, 1)
    train_total_loss /= max(n_train_batches, 1)

    # ---- Validate ----
    model.eval()
    val_task_losses = {f'task{i}': 0.0 for i in range(1, 6)}
    val_total_loss = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['input'].to(device)
            targets = {
                'density': batch['density'].unsqueeze(1).to(device),
                'susceptibility': batch['susceptibility'].unsqueeze(1).to(device),
                'structural_sim': batch['structural_sim'].unsqueeze(1).to(device),
            }
            preds = model(inputs)
            per_task, total_loss = criterion(preds, targets, model=None)

            for i in range(1, 6):
                val_task_losses[f'task{i}'] += (
                    per_task[f'task{i}'].item() if isinstance(per_task[f'task{i}'], torch.Tensor)
                    else per_task[f'task{i}']
                )
            val_total_loss += total_loss.item()
            n_val_batches += 1

    for i in range(1, 6):
        val_task_losses[f'task{i}'] /= max(n_val_batches, 1)
    val_total_loss /= max(n_val_batches, 1)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    # Record history
    train_epoch = {**train_task_losses, 'total': train_total_loss}
    val_epoch = {**val_task_losses, 'total': val_total_loss}
    history['train'].append(train_epoch)
    history['val'].append(val_epoch)

    # Incremental write to training_history.json after each epoch
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    # Log
    task_labels = ['T1(IndGrav)', 'T2(IndMag)', 'T3(Struct)', 'T4(JtGrav)', 'T5(JtMag)']
    train_str = " | ".join([f"{l}={train_task_losses[f'task{i+1}']:.4f}" 
                              for i, l in enumerate(task_labels)])
    val_str = " | ".join([f"{l}={val_task_losses[f'task{i+1}']:.4f}" 
                          for i, l in enumerate(task_labels)])
    print(f"Epoch [{epoch+1:3d}/{EPOCHS}] {epoch_time:.1f}s | LR={current_lr:.2e} | "
          f"Train(total={train_total_loss:.5f}): {train_str}\n"
          f"                                  Val(total={val_total_loss:.5f}):  {val_str}")

    # Save best model
    is_best = val_total_loss < best_val_loss
    if is_best:
        best_val_loss = val_total_loss
        early_stop_counter = 0
        ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
        }, ckpt_path)
        print(f"  ** New best model saved! Val loss: {best_val_loss:.6f} **")
    else:
        early_stop_counter += 1

    # Periodic checkpoint: every SAVE_EVERY epochs AND every 5 epochs (for resume safety)
    should_save = (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) % 5 == 0
    if should_save:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
        }, os.path.join(OUTPUT_DIR, 'checkpoints', f'checkpoint_epoch{epoch+1:03d}.pth'))

    # Release GPU memory after each epoch
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    # Early stopping
    if early_stop_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
        break

total_time = time.time() - total_start
print(f"\nTraining finished in {total_time/3600:.2f}h ({total_time:.0f}s)")
print(f"Best validation loss: {best_val_loss:.6f}")

In [ ]:
# Cell 7: Save best model & training history
import copy

# Save training history as JSON
history_path = os.path.join(OUTPUT_DIR, 'training_history.json')
save_json(history, history_path)
print(f"Training history saved to {history_path}")

# Confirm best model exists
best_model_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'best_model.pth')
assert os.path.exists(best_model_path), f"Best model not found at {best_model_path}"
print(f"Best model saved at: {best_model_path}")
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
print(f"  Best epoch: {ckpt['epoch']+1}")
print(f"  Best val loss: {ckpt['best_val_loss']:.6f}")

In [ ]:
# Cell 8: Evaluate metrics on test set (IoU, MSE, MAE, R2, SSIM, PSNR)
from src.evaluate import compute_all_metrics

# Load best model for evaluation
model.eval()

all_preds = {f'task{i}': [] for i in range(1, 6)}
all_targets = {'density': [], 'susceptibility': [], 'structural_sim': []}

print("Running evaluation on test set...")
with torch.no_grad():
    for idx, batch in enumerate(test_loader):
        inputs = batch['input'].to(device)
        targets = {
            'density': batch['density'].unsqueeze(1).to(device),
            'susceptibility': batch['susceptibility'].unsqueeze(1).to(device),
            'structural_sim': batch['structural_sim'].unsqueeze(1).to(device),
        }
        preds = model(inputs)

        for i in range(1, 6):
            all_preds[f'task{i}'].append(preds[f'task{i}'].cpu())
        for key in all_targets:
            all_targets[key].append(targets[key].cpu())

        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx+1}/{len(test_loader)} batches")

# Concatenate all batches
for i in range(1, 6):
    all_preds[f'task{i}'] = torch.cat(all_preds[f'task{i}'], dim=0)
for key in all_targets:
    all_targets[key] = torch.cat(all_targets[key], dim=0)

# Compute metrics (use first N samples for speed if needed)
N_EVAL = min(len(all_preds['task1']), 100)  # evaluate on up to 100 test samples
eval_preds = {k: v[:N_EVAL] for k, v in all_preds.items()}
eval_tgts = {k: v[:N_EVAL] for k, v in all_targets.items()}

print(f"\nComputing metrics on {N_EVAL} test samples...")
metrics = compute_all_metrics(eval_preds, eval_tgts, iou_threshold=0.5)

# Print results table
print("\n" + "=" * 75)
print(f"{'Task':<20s} {'IoU':>8s} {'MSE':>10s} {'MAE':>10s} {'R^2':>10s} {'SSIM':>8s} {'PSNR':>8s}")
print("-" * 75)
task_display_names = {
    'task1': 'Indep.Gravity',
    'task2': 'Indep.Magnetic',
    'task3': 'Struct.Sim',
    'task4': 'Joint.Gravity',
    'task5': 'Joint.Magnetic',
}
for tname, tmetrics in metrics.items():
    display_name = task_display_names.get(tname, tname)
    print(f"{display_name:<20s} {tmetrics['iou']:>8.4f} "
          f"{tmetrics['mse']:>10.6f} {tmetrics['mae']:>10.6f} "
          f"{tmetrics['r2']:>10.4f} {tmetrics['ssim']:>8.4f} {tmetrics['psnr']:>8.2f}")
print("=" * 75)

# Save metrics
metrics_path = os.path.join(OUTPUT_DIR, 'metrics.json')
save_json(metrics, metrics_path)
print(f"\nMetrics saved to {metrics_path}")

In [ ]:
# Cell 9: Result visualization (GT vs Pred inversion maps)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

os.makedirs(os.path.join(OUTPUT_DIR, 'inversion'), exist_ok=True)

# Pick a sample to visualize
sample_idx = 0

def plot_slice_comparison(pred_vol, gt_vol, title, depth_idx, save_path):
    """Plot a horizontal slice comparison: GT vs Pred side by side."""
    pred_slice = pred_vol[0, 0, depth_idx].cpu().numpy()  # (H, W)
    gt_slice = gt_vol[0, 0, depth_idx].cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    im0 = axes[0].imshow(gt_slice.T, origin='lower', cmap='viridis')
    axes[0].set_title(f'{title} - Ground Truth (depth slice {depth_idx})')
    plt.colorbar(im0, ax=axes[0], shrink=0.8)

    im1 = axes[1].imshow(pred_slice.T, origin='lower', cmap='viridis',
                          vmin=gt_slice.min(), vmax=gt_slice.max())
    axes[1].set_title(f'{title} - Prediction (depth slice {depth_idx})')
    plt.colorbar(im1, ax=axes[1], shrink=0.8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

# Visualize each task at a few depth slices
depth_slices = [3, 10, 17]  # approximate depths ~60m, 200m, 340m
task_viz_config = [
    ('task1', 'density', 'Independent Gravity Density'),
    ('task2', 'susceptibility', 'Independent Magnetic Suscept.'),
    ('task3', 'structural_sim', 'Structural Similarity'),
    ('task4', 'density', 'Joint Gravity Density'),
    ('task5', 'susceptibility', 'Joint Magnetic Suscept.'),
]

for tkey, tgt_key, title in task_viz_config:
    for didx in depth_slices:
        fname = f'inversion/{tkey}_slice_d{didx}.svg'
        save_path = os.path.join(OUTPUT_DIR, fname)
        plot_slice_comparison(
            eval_preds[tkey][sample_idx:sample_idx+1],
            eval_tgts[tgt_key][sample_idx:sample_idx+1],
            title, didx, save_path
        )
        print(f"Saved: {save_path}")

print("\nVisualization complete.")

In [ ]:
# Cell 10: Save all results to results/
import shutil

# Summary dict
summary = {
    'model': 'JointInversionNet (Fang et al. 2025 reproduction)',
    'epochs_trained': len(history['train']),
    'best_val_loss': float(best_val_loss),
    'test_metrics': {tk: {mk: float(mv) for mk, mv in tm.items()} 
                     for tk, tm in metrics.items()},
    'config': {
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'epochs': EPOCHS,
        'device': str(device),
    },
}

summary_path = os.path.join(OUTPUT_DIR, 'experiment_summary.json')
save_json(summary, summary_path)
print(f"Summary saved to {summary_path}")

# List all output files
print("\nOutput files in results/:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(files)[:20]:  # limit listing
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f"{subindent}{f} ({size_mb:.1f} MB)")

In [ ]:
# Cell 11: Final summary table
print("=" * 80)
print("  GRAVITY-MAGNETIC JOINT INVERSION - TRAINING COMPLETE")
print("=" * 80)
print(f"\n  Model:       JointInversionNet (U-Net3D + ASPP + 5 Task Heads)")
print(f"  Device:      {device}")
print(f"  Epochs:      {len(history['train'])}/{EPOCHS}")
print(f"  Best Val Loss: {best_val_loss:.6f}")
print(f"  Total Time:  {total_time/3600:.2f} hours")

print(f"\n  Test Set Metrics ({N_EVAL} samples):")
print(f"  {'Task':<22s} {'IoU':>8s} {'MSE':>10s} {'MAE':>10s} {'R^2':>10s} {'SSIM':>8s} {'PSNR':>8s}")
print(f"  {'-'*78}")
for tname, tmetrics in metrics.items():
    dn = task_display_names.get(tname, tname)
    print(f"  {dn:<22s} {tmetrics['iou']:>8.4f} "
          f"{tmetrics['mse']:>10.6f} {tmetrics['mae']:>10.6f} "
          f"{tmetrics['r2']:>10.4f} {tmetrics['ssim']:>8.4f} {tmetrics['psnr']:>8.2f}")

print(f"\n  Output dir:  {os.path.abspath(OUTPUT_DIR)}")
print("=" * 80)